All Implementations are based on the paper **Collapsed Variational Bayes Inference of Infinite Relational Model** [See PDF](https://arxiv.org/abs/1409.4757)

# Import Libraries and Modules

In [ ]:
import numpy as np  # Used for sampling and vectorization

# **Defining the single-domain IRM Generative Model**

Before we implement the Collapased Gibbs Sampler, we will define the generative model, which will produce data, that we want to perform inference on. The single-domain IRM Generative Model is defined by the following three parts:


*  **The Partitions (Chinese Restaurant Process)**
*  **The Block Probabilities (Beta Prior)**
*  **The Network Links (Bernoulli Likelihood)**



### **The Partitions (Chinese Restaurant Process)**

$$z_i|\alpha \sim \text{CRP}(\alpha) \qquad (11)$$


Because the CRP dynamically groups our $N$ nodes into clusters, it naturally determines the final number of occupied clusters, $K$. Once the CRP is finished seating all the "customers" (nodes), we can count how many "tables" (clusters) are occupied. That exact number becomes the $K$ we use to build our $K \times K$ matrix for the block probabilities, $\theta$.


$$p(z_i = k | z_{\backslash i}, \alpha) = \begin{cases} \frac{m_{\backslash i, k}}{N - 1 + \alpha} & m_{\backslash i, k} > 0 \text{ (existing cluster)} \\ \frac{\alpha}{N - 1 + \alpha} & m_{\backslash i, k} = 0 \text{ (new cluster)} \end{cases} \qquad (2)$$



In [ ]:
N = 50  # Example number of nodes
alpha = 1.0  # Concentration parameter

# Track the number of nodes in each cluster
cluster_counts = []

# Track the specific cluster assignment for each node i
Z = []

for i in range(N):
    # 1. Calculate weights and probabilities
    weights = cluster_counts + [alpha]
    total = sum(weights)
    probabilities = [w / total for w in weights]

    # 2. Make the choice
    # We give choice() a list of indices from 0 up to the number of weights
    options = range(len(weights))
    chosen_index = np.random.choice(options, p=probabilities)

    # 3. Record the assignment for node i
    Z.append(chosen_index)

    # 4. Update the counts for the next node
    if chosen_index == len(cluster_counts):
        # If the index equals the current number of clusters, it's a new table
        cluster_counts.append(1)
    else:
        # Otherwise, add one person to the chosen existing table
        cluster_counts[chosen_index] += 1

# The final number of clusters generated by the CRP
K = len(cluster_counts)

print(f"Generated {K} clusters for {N} nodes.")
print(f"Cluster counts: {cluster_counts}")
print(f"Node assignments (Z): {Z}")


Generated 4 clusters for 50 nodes.
Cluster counts: [14, 30, 5, 1]
Node assignments (Z): [np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(2), np.int64(1), np.int64(1), np.int64(3), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(2), np.int64(2), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(2), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(2), np.int64(1), np.int64(1), np.int64(0)]


### **The Block Probabilities (Beta Prior)**

$$\theta_{k,l}|a_{k,l}, b_{k,l} \sim \text{Beta}( a_{k,l}, b_{k,l} ) \qquad (10)$$

We can define a 2D array of theta values sampled from the beta distribution by defining the size of the matrix, given hyperparamters $a$ and $b$, this is faster than a for loop since we perform vectorization.

In [ ]:
# We set the hyperparameters a and b
a = 1
b = 1

# We define the matrix of theta values, sampled from the beta distribution
theta_matrix = np.random.beta(a, b, size=(K, K))

print(theta_matrix)

[[0.23762704 0.26451146 0.57234955 0.17775581]
 [0.57788907 0.28634494 0.13587143 0.79026589]
 [0.03526748 0.2866672  0.85132523 0.05414355]
 [0.31551507 0.50327883 0.07427005 0.2178365 ]]


### **The Network Links (Bernoulli Likelihood)**

$$x_{i, j}|Z, \{\theta\} \sim \text{Bernoulli}( \theta_{z_i,z_j} ) \qquad (12)$$

We now have $z$ whcich is our list of lenght $N$ where $z[i]$ tells us which cluster node $i$ belongs to.

And we have the theta_matrix which is our $K \times K$  matrix where theta_matrix[k,l] gives the probability of a link from clyster $k$ to cluster $l$

Now we need to generate our final $N \times N$ adjacency matrix, $X$, where X[i, j] = 1 if there is a link from node i to node j, and 0 otherwise.


In [ ]:
# Initialize an empty N x N matrix
X = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        # 1. Look up the probability for this specific pair of nodes
        p_link = theta_matrix[Z[i], Z[j]]

        # 2. Flip the coin! n=1 makes it a simple True/False (1 or 0) outcome
        X[i, j] = np.random.binomial(n=1, p=p_link)

print(X)

[[1. 0. 1. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [1. 1. 0. ... 0. 0. 0.]
 ...
 [0. 0. 1. ... 0. 0. 1.]
 [1. 1. 0. ... 1. 0. 1.]
 [0. 0. 0. ... 0. 1. 0.]]
